# 🚀 Amazon ML Challenge 2026: Full Test Submission Generator
## 📦 End-to-End Pipeline for All 1,732,544 Test Records (Google Colab Edition)

---

### 🌟 Purpose of This Notebook
This notebook produces the **final, full-scale submission files** required for the competition leaderboard and final ranking:
1. **`matching_results.tsv`**: Exactly **1,732,544 rows** (matching every single test Source 1 entity, zero missing rows).
2. **`candidate_pairs.tsv`**: The lean candidate baskets ($pprox 4 - 6$ candidates per entity) audited by Amazon for final rank.

```
┌────────────────────────────────────────────────────────────────────────┐
│  Step 1: Direct CDN Download to Colab (~15 seconds, 0% PC upload)       │
│  Step 2: Train LightGBM Model on Training Ground Truth                 │
│  Step 3: Country-by-Country Test Inference (France ➡️ US ➡️ India)      │
│  Step 4: Stream 1,732,544 Rows Directly to Disk (Memory < 4 GB)        │
│  Step 5: Run Official utils/validate_submission.py (Must print PASS)    │
│  Step 6: One-Click Browser Download to Your Laptop                      │
└────────────────────────────────────────────────────────────────────────┘
```


### 1. Step 1: Download & Extract Dataset in Colab (~15-20 Seconds)
Downloads the official archive directly from Unstop CDN into Colab's fast local SSD storage.


In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print(" Running inside Google Colab.")
    # Install fast C++ string similarity and LightGBM
    !pip install -q rapidfuzz lightgbm pyarrow
    
    # Download dataset directly from CDN if not already extracted
    if not os.path.exists("/content/student_resource/dataset"):
        if not os.path.exists("/content/6ab10eb3b23ba_student_resource.zip"):
            print("⚡ Downloading 1.09 GB archive directly from CDN...")
            !wget -q --show-progress https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip
        
        print("⚡ Extracting dataset into local Colab SSD...")
        !unzip -q /content/6ab10eb3b23ba_student_resource.zip -d /content/
        print(" Extraction complete!")
    else:
        print(" Dataset already ready in SSD.")
        
    DATA_DIR = "/content/student_resource/dataset"
    UTILS_DIR = "/content/student_resource/utils"
else:
    print("💻 Running in local environment.")
    DATA_DIR = os.path.abspath(os.path.join("dataset_folder", "student_resource", "dataset"))
    UTILS_DIR = os.path.abspath(os.path.join("dataset_folder", "student_resource", "utils"))

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

print("Train Directory:", TRAIN_DIR)
print("Test Directory :", TEST_DIR)


### 2. High-Performance Libraries & Hardware
Loads **RapidFuzz** for compiled C++ string comparisons and initializes multi-core parallelism.


In [ ]:
import gc
import re
import csv
import time
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# RapidFuzz for high-speed string matching (with pure Python fallback)
try:
    from rapidfuzz import fuzz
    print(" RapidFuzz C++ engine loaded.")
except ImportError:
    import difflib
    print("⚠️ RapidFuzz not found. Using standard library fallback.")
    class DummyFuzz:
        @staticmethod
        def ratio(s1, s2): return difflib.SequenceMatcher(None, s1, s2).ratio() * 100.0
        @staticmethod
        def token_sort_ratio(s1, s2):
            t1, t2 = ' '.join(sorted(s1.split())), ' '.join(sorted(s2.split()))
            return difflib.SequenceMatcher(None, t1, t2).ratio() * 100.0
        @staticmethod
        def token_set_ratio(s1, s2):
            t1, t2 = set(s1.split()), set(s2.split())
            inter = ' '.join(sorted(t1 & t2))
            return max(difflib.SequenceMatcher(None, inter, ' '.join(sorted(t1))).ratio() * 100.0,
                       difflib.SequenceMatcher(None, inter, ' '.join(sorted(t2))).ratio() * 100.0)
    fuzz = DummyFuzz()

from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
print("Libraries initialized.")


### 3. Dual-Track Preprocessing Engine
* **Cleaned Track**: Used for finding candidates (accents stripped, legal tags removed, road words standardized).
* **Raw Track**: Preserved for the ML judge + extracted legal tag (`LLC`, `Inc`, `SARL`) as a decisive tie-breaker.


In [ ]:
# 1. Accent & Diacritic Stripping (e.g. Bárbershop -> Barbershop)
def strip_accents(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

# 2. Company Legal Suffix Categorization (US, India, and France)
TAG_PATTERNS = {
    'llc': r'\b(limited liability company|llc)\b',
    'inc': r'\b(incorporated|inc|corporation|corp)\b',
    'ltd': r'\b(private limited|pvt ltd|pvt\. ltd\.|limited|ltd|llp)\b',
    'sarl': r'\b(societe a responsabilite limitee|sarl|sasu|sas|eurl|sci)\b',
    'assoc': r'\b(association|assoc|cie|et fils|& fils)\b'
}
ALL_TAGS_REGEX = re.compile('|'.join(TAG_PATTERNS.values()), flags=re.IGNORECASE)

def extract_legal_tag(name: str) -> str:
    if not isinstance(name, str):
        return "none"
    name_lower = strip_accents(name).lower()
    for tag_name, pat in TAG_PATTERNS.items():
        if re.search(pat, name_lower):
            return tag_name
    return "none"

# 3. Clean Business Name
def clean_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    name = strip_accents(name).lower()
    name = re.sub(r'(@\w+|https?://\S+|www\.\S+|\.com|\.org|\.in|\.fr)', ' ', name)
    name = ALL_TAGS_REGEX.sub(' ', name)
    name = re.sub(r'\b(d/?b/?a|trading as|t/a)\b.*', ' ', name)
    name = re.sub(r'\([\w\s:]+\)|\[[\w\s:]+\]', ' ', name)
    name = re.sub(r'(?<=[a-z])1\b', 'l', name) # Fix OCR typo (chape1 -> chapel)
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    return ' '.join(name.split())

# 4. Standardize Address & Extract House Numbers
ROAD_MAP = {
    r'\b(street|str)\b': 'st',
    r'\b(road)\b': 'rd',
    r'\b(avenue|ave\.)\b': 'ave',
    r'\b(boulevard|blvd\.)\b': 'blvd',
    r'\b(drive|dr\.)\b': 'dr',
    r'\b(lane|ln\.)\b': 'ln',
    r'\b(highway|hwy\.)\b': 'hwy',
    r'\b(rue|r\.)\b': 'rue',
    r'\b(allee|all\.)\b': 'allee'
}

def clean_address(addr: str) -> str:
    if not isinstance(addr, str):
        return ""
    addr = strip_accents(addr).lower()
    for pat, rep in ROAD_MAP.items():
        addr = re.sub(pat, rep, addr)
    addr = re.sub(r'\b(po box|pmb|hn|plot no|house no|flat no|unit|apt|suite)\b[\s\d-]*', ' ', addr)
    addr = re.sub(r'[^a-z0-9\s,]', ' ', addr)
    return ' '.join(addr.split())

def extract_numbers(addr: str) -> set:
    if not isinstance(addr, str):
        return set()
    nums = re.findall(r'\b\d+\b', addr)
    return {str(int(n)) for n in nums if len(n) <= 6}

print("Preprocessing functions ready.")


### 4. Precision Blocking Engine (Lean Candidate Baskets)
Caps candidate baskets at **maximum 8 candidates per entity** to satisfy Amazon's candidate set ranking criterion.


In [ ]:
class PrecisionBlockingEngine:
    def __init__(self, max_candidates=8):
        self.indices = defaultdict(set)
        self.records = {}
        self.max_candidates = max_candidates
        
    def _make_keys(self, c_name, c_addr, nums):
        keys = []
        toks = c_name.split()
        if toks:
            if len(toks[0]) >= 3:
                addr_prefix = c_addr[:3] if len(c_addr) >= 3 else "xx"
                keys.append(f"k1:{toks[0]}_{addr_prefix}")
            if len(toks) >= 2:
                keys.append(f"k2:{'-'.join(sorted(toks[:2]))}")
        prefix2 = c_name[:2] if len(c_name) >= 2 else "xx"
        for n in nums:
            keys.append(f"k3:{n}_{prefix2}")
        return keys

    def add_records(self, df_cands):
        for _, row in df_cands.iterrows():
            eid = row['entity_id']
            c_name = clean_name(row['business_name'])
            c_addr = clean_address(row['business_address'])
            nums = extract_numbers(row['business_address'])
            tag = extract_legal_tag(row['business_name'])
            
            self.records[eid] = (row['business_name'], c_name, tag, row['business_address'], c_addr, nums)
            for k in self._make_keys(c_name, c_addr, nums):
                self.indices[k].add(eid)

    def get_candidates(self, c_name, c_addr, nums):
        keys = self._make_keys(c_name, c_addr, nums)
        counts = Counter()
        for k in keys:
            for eid in self.indices.get(k, set()):
                counts[eid] += 1
        return [eid for eid, _ in counts.most_common(self.max_candidates)]

# 15 Pairwise Features
def compute_clues(s1_raw_name, s1_clean_name, s1_tag, s1_raw_addr, s1_clean_addr, s1_nums,
                  cand_raw_name, cand_clean_name, cand_tag, cand_raw_addr, cand_clean_addr, cand_nums,
                  cand_id):
    feats = {}
    feats['clean_name_sort'] = fuzz.token_sort_ratio(s1_clean_name, cand_clean_name) / 100.0
    feats['clean_name_lev']  = fuzz.ratio(s1_clean_name, cand_clean_name) / 100.0
    feats['clean_name_set']  = fuzz.token_set_ratio(s1_clean_name, cand_clean_name) / 100.0
    
    s1_toks = set(s1_clean_name.split())
    cand_toks = set(cand_clean_name.split())
    feats['first_word_match'] = 1.0 if (s1_toks and cand_toks and s1_clean_name.split()[0] == cand_clean_name.split()[0]) else 0.0
    
    feats['raw_name_sort']   = fuzz.token_sort_ratio(s1_raw_name, cand_raw_name) / 100.0
    feats['raw_name_lev']    = fuzz.ratio(s1_raw_name, cand_raw_name) / 100.0
    
    # Tag Match
    if s1_tag == cand_tag and s1_tag != 'none':
        feats['tag_score'] = 1.0
    elif s1_tag == 'none' or cand_tag == 'none':
        feats['tag_score'] = 0.5
    else:
        feats['tag_score'] = 0.0

    # Address & Numbers
    feats['has_empty_addr']  = 1.0 if (not s1_clean_addr or not cand_clean_addr) else 0.0
    if feats['has_empty_addr']:
        feats['addr_token_sort'] = 0.0
        feats['number_jaccard']  = 0.0
        feats['number_exact_match'] = 0.0
    else:
        feats['addr_token_sort'] = fuzz.token_sort_ratio(s1_clean_addr, cand_clean_addr) / 100.0
        num_union = s1_nums | cand_nums
        feats['number_jaccard'] = len(s1_nums & cand_nums) / len(num_union) if num_union else 0.5
        feats['number_exact_match'] = 1.0 if (s1_nums and cand_nums and s1_nums == cand_nums) else 0.0

    feats['is_source2'] = 1.0 if cand_id.startswith('S2-') else 0.0
    feats['is_source3'] = 1.0 if cand_id.startswith('S3-') else 0.0
    return feats


### 5. Train the LightGBM Judge
We train the model on a representative slice of the training data (e.g. 10,000 S1 records with positive matches and hard negative candidate pairs).


In [ ]:
print("Loading training slice to train LightGBM judge...")
TRAIN_SAMPLE_SIZE = 10000

s1_train_rows = []
with open(os.path.join(TRAIN_DIR, "train_source1.tsv"), "r", encoding="utf-8") as f:
    r = csv.reader(f, delimiter="\t")
    hdr = next(r)
    for i, row in enumerate(r):
        s1_train_rows.append(row)
        if i + 1 >= TRAIN_SAMPLE_SIZE:
            break

df_train_s1 = pd.DataFrame(s1_train_rows, columns=hdr)
sampled_train_s1_ids = set(df_train_s1['entity_id'])

# Ground truth map
train_gt = {}
needed_train_cands = set()
with open(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"), "r", encoding="utf-8") as f:
    r = csv.reader(f, delimiter="\t")
    next(r)
    for row in r:
        if row[0] in sampled_train_s1_ids:
            m = row[1].split(",") if len(row) > 1 and row[1].strip() else []
            train_gt[row[0]] = set(m)
            for cid in m:
                needed_train_cands.add(cid)

# Candidate pool for training
def read_train_cands(fname, needed_ids, max_noise=25000):
    rows = []
    noise = 0
    with open(os.path.join(TRAIN_DIR, fname), "r", encoding="utf-8") as f:
        r = csv.reader(f, delimiter="\t")
        h = next(r)
        for row in r:
            if row[0] in needed_ids:
                rows.append(row)
            elif noise < max_noise:
                rows.append(row)
                noise += 1
    return pd.DataFrame(rows, columns=h)

df_tc2 = read_train_cands("train_source2.tsv", needed_train_cands)
df_tc3 = read_train_cands("train_source3.tsv", needed_train_cands)
df_train_cands = pd.concat([df_tc2, df_tc3], ignore_index=True)

# Build training blocking engine
train_blocking = PrecisionBlockingEngine(max_candidates=8)
train_blocking.add_records(df_train_cands)

# Build feature matrix
X_train_rows, y_train_labels = [], []
for _, row in df_train_s1.iterrows():
    s1_id = row['entity_id']
    raw_name = row['business_name']
    c_name = clean_name(raw_name)
    tag = extract_legal_tag(raw_name)
    raw_addr = row['business_address']
    c_addr = clean_address(raw_addr)
    nums = extract_numbers(raw_addr)
    
    cands = train_blocking.get_candidates(c_name, c_addr, nums)
    true_set = train_gt.get(s1_id, set())
    
    for cid in cands:
        cand_info = train_blocking.records.get(cid)
        if not cand_info:
            continue
        feats = compute_clues(
            raw_name, c_name, tag, raw_addr, c_addr, nums,
            cand_info[0], cand_info[1], cand_info[2], cand_info[3], cand_info[4], cand_info[5],
            cid
        )
        label = 1 if cid in true_set else 0
        X_train_rows.append(feats)
        y_train_labels.append(label)

df_X_train = pd.DataFrame(X_train_rows)
y_train_arr = np.array(y_train_labels)

print(f"Training LightGBM on {len(df_X_train):,} candidate pairs...")
model = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
model.fit(df_X_train, y_train_arr)
print(" LightGBM model trained successfully!")

# Free training slice memory
del df_train_s1, df_train_cands, df_tc2, df_tc3, train_blocking, df_X_train, y_train_arr
gc.collect()


### 6. Full Test Inference: Country-by-Country Streaming
To process all **1,732,544 test entities** without exceeding RAM limits, we process one country at a time:
1. **France** (~259k entities)
2. **US** (~663k entities)
3. **India** (~810k entities)

Every single test Source 1 entity is streamed, evaluated, and immediately appended to disk!


In [ ]:
# Output directory & files
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

matching_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

# Initialize files with headers
with open(matching_path, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")

with open(candidate_path, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")

# Decision threshold tuned for high precision (F_0.5)
DECISION_THRESHOLD = 0.70

countries_to_process = ['France', 'US', 'India']
total_entities_processed = 0
start_time_all = time.time()

for country in countries_to_process:
    print("=" * 65)
    print(f" Processing Country: {country.upper()}...")
    print("=" * 65)
    
    # 1. Load S2 and S3 for this country only
    cands_country = []
    for fname in ["test_source2.tsv", "test_source3.tsv"]:
        p = os.path.join(TEST_DIR, fname)
        with open(p, "r", encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="\t")
            hdr = next(reader)
            for row in reader:
                if row[3] == country:
                    cands_country.append(row)
                    
    df_cands_country = pd.DataFrame(cands_country, columns=['entity_id', 'business_name', 'business_address', 'country'])
    print(f"  Loaded {len(df_cands_country):,} candidates for {country}.")
    
    # 2. Build blocking engine for this country
    engine = PrecisionBlockingEngine(max_candidates=8)
    engine.add_records(df_cands_country)
    del cands_country, df_cands_country
    gc.collect()
    
    # 3. Stream test_source1 for this country and write directly to disk
    country_s1_count = 0
    t0_country = time.time()
    
    with open(os.path.join(TEST_DIR, "test_source1.tsv"), "r", encoding="utf-8") as f_s1, \
         open(matching_path, "a", encoding="utf-8") as f_match, \
         open(candidate_path, "a", encoding="utf-8") as f_cand:
        
        reader = csv.reader(f_s1, delimiter="\t")
        next(reader)
        
        batch_rows = []
        for row in reader:
            if row[3] != country:
                continue
                
            country_s1_count += 1
            total_entities_processed += 1
            s1_id = row[0]
            raw_name = row[1]
            c_name = clean_name(raw_name)
            tag = extract_legal_tag(raw_name)
            raw_addr = row[2]
            c_addr = clean_address(raw_addr)
            nums = extract_numbers(raw_addr)
            
            cands = engine.get_candidates(c_name, c_addr, nums)
            
            if not cands:
                # Singleton (no candidate basket)
                f_match.write(f"{s1_id}\t\n")
                f_cand.write(f"{s1_id}\t\n")
            else:
                # Score candidates with model
                feats_list = []
                for cid in cands:
                    c_info = engine.records[cid]
                    feats = compute_clues(
                        raw_name, c_name, tag, raw_addr, c_addr, nums,
                        c_info[0], c_info[1], c_info[2], c_info[3], c_info[4], c_info[5],
                        cid
                    )
                    feats_list.append(feats)
                    
                df_pair = pd.DataFrame(feats_list)
                probs = model.predict_proba(df_pair)[:, 1]
                
                matched_ids = [cid for cid, p in zip(cands, probs) if p >= DECISION_THRESHOLD]
                
                f_match.write(f"{s1_id}\t{','.join(matched_ids)}\n")
                f_cand.write(f"{s1_id}\t{','.join(cands)}\n")
                
            if country_s1_count % 100000 == 0:
                print(f"  Processed {country_s1_count:,} entities in {country}...")
                
    print(f" Finished {country}: {country_s1_count:,} entities processed in {time.time() - t0_country:.2f}s.")
    
    # Free memory before next country
    del engine
    gc.collect()

print("=" * 65)
print(f"🎉 INFERENCE COMPLETE FOR ALL {total_entities_processed:,} TEST ENTITIES!")
print(f"Total time elapsed: {(time.time() - start_time_all)/60:.2f} minutes.")
print("=" * 65)


### 7. Run Official Validator (Guaranteeing `PASS`)
Before submitting to the portal, we run the competition script `utils/validate_submission.py`. 
It must print **`PASS`** with zero missing rows!


In [ ]:
# Run official validator script
validator_script = os.path.join(UTILS_DIR, "validate_submission.py")
print("Executing official competition validator...")

!python "{validator_script}" \
    --matching "{matching_path}" \
    --candidate "{candidate_path}" \
    --test-dir "{TEST_DIR}"


### 8. One-Click Download to Your Laptop
Zips both validated output files and triggers an automatic browser download.


In [ ]:
if IN_COLAB:
    print("Preparing download archive...")
    !zip -q final_submission.zip output/matching_results.tsv output/candidate_pairs.tsv
    from google.colab import files
    files.download("final_submission.zip")
    print(" Download prompt triggered in your browser!")
else:
    print(f"Files are ready in: {os.path.abspath('output')}")
